In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Geometry-V7 R1B
Run the exact-bound truth-utility and under-correction canary once, then publish its create-only result package.

In [ ]:
import re
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
APPROVED_EXACT = 'PENDING_AFTER_GEOMETRY_V7_R1B_PUSH'
if re.fullmatch(r'[0-9a-f]{40}', APPROVED_EXACT) is None:
    raise RuntimeError('Bind the approved pushed Geometry-V7 R1B exact before execution')
checkout = Path('/content/CEG-WM')
if checkout.exists():
    raise FileExistsError(f'create-only checkout already exists: {checkout}')
subprocess.run(['git', 'clone', REPO_URL, str(checkout)], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', APPROVED_EXACT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(checkout)], check=True)

In [ ]:
import os
import torch
from google.colab import userdata

assert torch.cuda.is_available(), 'GPU required; no R1B result on CPU'
R0_ARTIFACT_ROOT = Path('/content/drive/MyDrive/CEG-WM/Geometry-V7/4f0bf1560805672f786dc86dd50d793aec18aae7/r0-f1')
R1A_ARTIFACT_ROOT = Path('/content/drive/MyDrive/CEG-WM/Geometry-V7/ac590330e91aacf4b3283df1e94572a0e4f983a0/r1a-f2')
LOCAL_RESULT_DIR = Path('/content/geometry_v7_r1b_result')
DRIVE_RESULT_DIR = Path('/content/drive/MyDrive/CEG-WM/Geometry-V7') / APPROVED_EXACT / 'r1b'
if not R0_ARTIFACT_ROOT.is_dir() or not R1A_ARTIFACT_ROOT.is_dir():
    raise FileNotFoundError('fixed accepted R0/R1A input artifact is absent')
if LOCAL_RESULT_DIR.exists():
    raise FileExistsError('create-only local R1B result already exists')
if DRIVE_RESULT_DIR.exists():
    raise FileExistsError(f'create-only Drive result already exists: {DRIVE_RESULT_DIR}')
hf_token = userdata.get('HF_TOKEN')
root_key = userdata.get('CEG_WM_ROOT_KEY')
if not hf_token or not root_key:
    raise RuntimeError('HF_TOKEN and CEG_WM_ROOT_KEY Colab secrets are required')
markers = ('TOKEN', 'KEY', 'SECRET', 'PASSWORD', 'CREDENTIAL')
runner_env = {
    name: value for name, value in os.environ.items()
    if not any(marker in name.upper() for marker in markers)
}
runner_env['HF_TOKEN'] = hf_token
runner_env['CEG_WM_ROOT_KEY'] = root_key
command = [
    sys.executable, '-m', 'experiments.run_geometry_v7_r1b',
    '--repo-root', str(checkout), '--expected-exact', APPROVED_EXACT,
    '--r0-artifact-root', str(R0_ARTIFACT_ROOT),
    '--r1a-artifact-root', str(R1A_ARTIFACT_ROOT),
    '--result-dir', str(LOCAL_RESULT_DIR),
]
try:
    completed = subprocess.run(
        command, cwd=checkout, env=runner_env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=False,
    )
finally:
    runner_env.clear()
    hf_token = ''
    root_key = ''
print(completed.stdout.strip())
if completed.returncode not in (0, 2):
    raise RuntimeError('Geometry-V7 R1B runner stopped before a complete package')
if not (LOCAL_RESULT_DIR / 'result.json').is_file():
    raise RuntimeError('Geometry-V7 R1B complete result package is absent')

Publish only after the local create-only package contains `result.json`.

In [ ]:
import shutil

if not (LOCAL_RESULT_DIR / 'result.json').is_file():
    raise RuntimeError('Run the bound real R1B producer before publication')
if DRIVE_RESULT_DIR.exists():
    raise FileExistsError(f'create-only Drive result already exists: {DRIVE_RESULT_DIR}')
DRIVE_RESULT_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_RESULT_DIR, DRIVE_RESULT_DIR)
print(DRIVE_RESULT_DIR)